#### **Step 1: Import Required Libraries**


In [1]:
from pyspark.sql.functions import (col, lower, upper, when, expr, regexp_replace, to_timestamp, coalesce, isnan, count, trim, round, to_date,
 date_format, bround, lit, initcap,array_distinct, sort_array, transform, concat_ws, current_date, datediff, abs, udf, current_timestamp, row_number, split, length)
from pyspark.sql.window import Window
from pyspark.sql.types import FloatType, TimestampType, DateType, StringType
from delta.tables import DeltaTable
from datetime import datetime

StatementMeta(, e5328917-e465-49a1-802d-f08fe18f3178, 3, Finished, Available, Finished)

#### **Step 2: Load Raw Data from Lakehouse (Bronze Layer)**

In [91]:
# Create path variable
raw_transaction_path = "Files/cdp_bronze/oracle/raw/users"
df_raw_users = spark.read.format("parquet").load(raw_transaction_path)
display(df_raw_users.limit(10))

StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 93, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, b25c1360-41b2-4443-809a-f2cadd87923d)

### **Column-by-Column Transformation Plan**
user_id
- Current: Clean and consistent (e.g., BB1000xxxx).
- Action: No transformation needed.
- Final: Keep as-is.

first_name, last_name
- Kept as-is since values are already consistent and non-null in most cases.

#### **Step 3: Inspect the users dataframe for data quality checks**

In [92]:
# Select specific columns and count nulls
null_counts = df_raw_users.select(
    [count(when(col(c).isNull(), c)).alias(c) for c in 
     ['user_id', 'username', 'email', 'first_name', 'last_name', 'date_of_birth', 'registration_date', 'country_code', 'kyc_status', 'account_status']
    ]
)

# Display results
null_counts.show()

StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 94, Finished, Available, Finished)

+-------+--------+-----+----------+---------+-------------+-----------------+------------+----------+--------------+
|user_id|username|email|first_name|last_name|date_of_birth|registration_date|country_code|kyc_status|account_status|
+-------+--------+-----+----------+---------+-------------+-----------------+------------+----------+--------------+
|      0|     912| 1446|         0|        0|            0|                0|           0|      3048|          1590|
+-------+--------+-----+----------+---------+-------------+-----------------+------------+----------+--------------+



#### **Step 4: Data Transformation to username & email columns**

username
Issue: 
- Some values were null, others had inconsistent casing or whitespace.
- If username is null, it is replaced with the corresponding first_name

Transformation:
- Trimmed whitespaces.
- Converted to lowercase.
- Retained null where username was missing.

email
Issues:
- Some emails were empty strings or missing.
- Some were malformed (e.g., missing @ or domain).

Transformation:
- Trimmed and lowercased all emails.
- Replaced malformed or empty values with null.


In [93]:

# Clean the 'username' column
# - If 'username' is null, replace it with the corresponding 'first_name'
# - Otherwise, trim and lowercase the 'username' value

df_cleaned_users = df_raw_users.withColumn(
    "username",
    when(col("username").isNull(), lower(trim(col("first_name"))))  # fallback to 'first_name'
    .otherwise(lower(trim(col("username"))))  # clean existing username
)

# Define email regex pattern for validation
email_regex = r'^[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}$'

# Clean the 'email' column
# - Set empty or invalid emails to None
# - Trim and lowercase valid email addresses

df_cleaned_users = df_cleaned_users.withColumn(
    "email",
    when(trim(col("email")) == "", None)  # Empty string to null
    .when(~col("email").rlike(email_regex), None)  # Invalid email to null
    .otherwise(lower(trim(col("email"))))  # Clean valid emails
)

# Preview the cleaned DataFrame
df_cleaned_users.show(10, truncate=False)


StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 95, Finished, Available, Finished)

+----------+----------------+-----------------------------+----------+---------+-------------+-----------------+------------+----------+--------------+
|USER_ID   |username        |email                        |FIRST_NAME|LAST_NAME|DATE_OF_BIRTH|REGISTRATION_DATE|COUNTRY_CODE|KYC_STATUS|ACCOUNT_STATUS|
+----------+----------------+-----------------------------+----------+---------+-------------+-----------------+------------+----------+--------------+
|BB10000344|kim78           |thompsoncatherine@example.com|Patricia  |Gonzalez |1999-08-29   |2021-03-18       |CU          |Pending   |Pending       |
|BB10000069|andrea47        |omartinez@example.net        |Jennifer  |Collins  |1972-03-14   |2026-04-06       |GY          |Pending   |Pending       |
|BB10000104|reeddaniel      |trose@example.org            |Randall   |Greene   |N/A          |2014-02-16       |GN          |Pending   |Pending       |
|BB10000119|jenniferwilliams|erica37@example.net          |Karen     |Bowers   |1955-09-

#### **Step 5: Data Transformation to date_of_birth & registration_date columns**
Issues:
- Multiple inconsistent formats (e.g., 31/04/1985, Feb 30 1992).
- Some dates were invalid or impossible.
- Some were in the future.

Transformation:
- Parsed using multiple known date formats.
- Replaced unparsable or future dates with null.

In [94]:
# Helper function to parse multiple date formats consistently
def parse_date(column):
    return coalesce(
        to_date(column, "yyyy-MM-dd"),
        to_date(column, "MM-dd-yyyy"),
        to_date(column, "dd/MM/yyyy"),
        to_date(column, "MMM dd yyyy"),
        to_date(column, "MMM dd yyyy HH:mm"),
        to_date(column, "dd MMM yyyy")
    )

# Parse and standardize 'date_of_birth' and 'registration_date' columns
df_cleaned_users = (
    df_cleaned_users
    .withColumn("dob_parsed", parse_date(col("date_of_birth")))
    .withColumn("reg_parsed", parse_date(col("registration_date")))
    .withColumn("date_of_birth", col("dob_parsed"))
    .withColumn("registration_date", col("reg_parsed"))
    .drop("dob_parsed", "reg_parsed")
)

# Define the current date for validation
today = current_date()

# Remove records with null or future dates
df_cleaned_users = df_cleaned_users.filter(
    (col("date_of_birth").isNotNull()) &
    (col("registration_date").isNotNull()) &
    (col("date_of_birth") < today) &
    (col("registration_date") < today)
)

# Ensure registration_date is not earlier than date_of_birth
df_cleaned_users = df_cleaned_users.filter(
    col("registration_date") >= col("date_of_birth")
)

# Retain users who are 18 years or older
df_cleaned_users = df_cleaned_users.filter(
    datediff(today, col("date_of_birth")) >= 18 * 365
)

# Optional: Re-normalize just to ensure date-only format
df_cleaned_users = df_cleaned_users.withColumn("date_of_birth", to_date(col("date_of_birth"))) \
                                   .withColumn("registration_date", to_date(col("registration_date")))

# Display sample of the cleaned dataset
df_cleaned_users.show(20, truncate=False)


StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 96, Finished, Available, Finished)

+----------+----------------+-----------------------------+----------+---------+-------------+-----------------+------------+----------+--------------+
|USER_ID   |username        |email                        |FIRST_NAME|LAST_NAME|date_of_birth|registration_date|COUNTRY_CODE|KYC_STATUS|ACCOUNT_STATUS|
+----------+----------------+-----------------------------+----------+---------+-------------+-----------------+------------+----------+--------------+
|BB10000344|kim78           |thompsoncatherine@example.com|Patricia  |Gonzalez |1999-08-29   |2021-03-18       |CU          |Pending   |Pending       |
|BB10000119|jenniferwilliams|erica37@example.net          |Karen     |Bowers   |1955-09-30   |2009-06-30       |JM          |Pending   |Pending       |
|BB10000168|gina            |contrerassteven@example.org  |Gina      |Leblanc  |1985-07-14   |2018-08-24       |NA          |Pending   |Pending       |
|BB10000173|hansenemily     |jennifer64@example.org       |Jeffery   |Jones    |1976-09-

#### **Step 6: Data Transformation to country_code column**
Issues:
- Some values were non-standard (e.g., "U.K" instead of "UK").
- Some had lowercase or mixed-case characters.

Transformation:
- Trimmed and converted to uppercase.
- Replaced "U.K" with standard "UK".

In [95]:
# Standardize country_code column
df_cleaned_users = df_cleaned_users.withColumn(
    "country_code",
    upper(
        regexp_replace(
            lower(trim(col("country_code"))),
            r"[.,]",
            ""
        )
    )
)

StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 97, Finished, Available, Finished)

#### **Step 7: Data Transformation to kyc_status & account_status columns**
kyc_status
Issues:
- Inconsistent casing (e.g., verified, Verified, VERIFIED).
- Presence of unknown or empty statuses.

Transformation:
- Standardized to: Pending, Verified, Rejected, or Unknown.

account_status
Issues:
- Presence of values like "active123", blanks, or inconsistent casing.

Transformation:
- Mapped valid values to: Active, Suspended, Closed, Pending, Unknown.

In [96]:
# Define valid KYC status mappings
valid_kyc = {
    "pending":   "Pending",
    "verified":  "Verified",
    "rejected":  "Rejected"
}

# Define valid account status mappings
valid_account = {
    "active":     "Active",
    "suspended":  "Suspended",
    "closed":     "Closed",
    "pending":    "Pending"
}

# Standardize and validate 'kyc_status' values
# - Trims whitespace and converts to lowercase
# - Maps known statuses to their proper format
# - Assigns 'Unknown' to invalid or unrecognized values
df_cleaned_users = df_cleaned_users.withColumn(
    "kyc_status",
    when(
        lower(trim(col("kyc_status"))).isin(list(valid_kyc.keys())),
        expr("map('pending','Pending','verified','Verified','rejected','Rejected')[lower(trim(kyc_status))]")
    ).otherwise("Unknown")
)

# Standardize and validate 'account_status' values
# - Same logic as above, applied to account statuses
df_cleaned_users = df_cleaned_users.withColumn(
    "account_status",
    when(
        lower(trim(col("account_status"))).isin(list(valid_account.keys())),
        expr("map('active','Active','suspended','Suspended','closed','Closed','pending','Pending')[lower(trim(account_status))]")
    ).otherwise("Unknown")
)

# Select relevant and cleaned columns for final dataset
df_cleaned_users = df_cleaned_users.select(
    "user_id", "username", "email",
    "first_name", "last_name",
    "date_of_birth", "registration_date",
    "country_code", "kyc_status", "account_status"
)

# Display a sample of cleaned user records
display(df_cleaned_users.limit(10))


StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 98, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, 1e16bb37-61d0-4260-a59d-ea244e0dd46d)

#### **Step 8: Enforcing KYC ↔ Account Status Relationships and Deduplication**
Purpose:
Ensure logical consistency between KYC and account statuses, and remove outdated or duplicate user records.

Enforced Rules:
- If kyc_status = Rejected → force account_status = Suspended.
- If kyc_status = Verified and account_status = Unknown → default account_status = Active.
- If kyc_status = Pending and account_status = Unknown → default account_status = Pending.
- Default all other invalid/missing kyc_status to "Pending" to allow for verification later.

Deduplication Logic:
- Added load_timestamp to each record.
- Applied windowing (ROW_NUMBER) to retain latest record per user_id.
- Dropped older duplicates based on load_timestamp.

Result:
- Clean, valid user records.
- No Unknown combinations that break business rules.
- Only the most recent entry per user is retained for loading to the silver layer.

In [97]:
# Add a 'load_timestamp' column to track when the data was loaded
load_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_cleaned_users = df_cleaned_users.withColumn("load_timestamp", lit(load_ts))

# Enforce logical relationships between KYC and Account status
df_cleaned_users = df_cleaned_users.withColumn(
    "kyc_status",
    when(col("kyc_status").isin("Pending", "Verified", "Rejected"), col("kyc_status"))
    .otherwise("Pending")  # Replace 'Unknown' or nulls with 'Pending' as a safe default
)

df_cleaned_users = df_cleaned_users.withColumn(
    "account_status",
    when((col("kyc_status") == "Rejected") & (~col("account_status").isin("Suspended", "Closed")), "Suspended")
    .when((col("kyc_status") == "Verified") & (col("account_status") == "Unknown"), "Active")
    .when((col("kyc_status") == "Pending") & (col("account_status") == "Unknown"), "Pending")
    .when(col("account_status").isin("Active", "Suspended", "Closed", "Pending"), col("account_status"))
    .otherwise("Pending")  # Replace unknowns or invalids with 'Pending'
)

# Deduplicate records based on latest load_timestamp per user_id
window_spec = Window.partitionBy("user_id").orderBy(col("load_timestamp").desc())

df_cleaned_users = df_cleaned_users.withColumn("row_num", row_number().over(window_spec)) \
                                   .filter(col("row_num") == 1) \
                                   .drop("row_num")

# Cleaned DataFrame with consistent statuses and unique users
df_cleaned_users.show(truncate=False)


StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 99, Finished, Available, Finished)

+----------+----------------+----------------------------+----------+----------+-------------+-----------------+------------+----------+--------------+-------------------+
|user_id   |username        |email                       |first_name|last_name |date_of_birth|registration_date|country_code|kyc_status|account_status|load_timestamp     |
+----------+----------------+----------------------------+----------+----------+-------------+-----------------+------------+----------+--------------+-------------------+
|BB10000005|oramirez        |francisco53@example.net     |Jacqueline|Sutton    |1998-03-27   |2024-02-18       |IQ          |Pending   |Active        |2025-07-08 03:40:26|
|BB10000010|shawnmckay      |teresa28@example.org        |Kenneth   |Fowler    |1965-10-13   |1995-12-22       |YE          |Pending   |Active        |2025-07-08 03:40:26|
|BB10000014|pcarney         |samuel87@example.org        |Jeffrey   |Morgan    |1987-10-24   |2019-09-27       |HU          |Verified  |Acti

#### **Step 9: Data Quality Checks**
- The script performs data quality checks on the df_cleaned_users DataFrame by validating key fields like user_id, email, date_of_birth, and categorical columns. It flags issues such as nulls, invalid formats, future or underage dates, and unexpected values. Then, it summarizes the number of violations for each rule and reports whether the data meets defined quality thresholds—helping decide if the data is safe to load into the Silver layer.

| Check                      | Details                                     |
| -------------------------- | ------------------------------------------- |
| `null_user_id`             | Should never be null or blank               |
| `null_username`            | Must not be null                            |
| `invalid_email`            | Email should contain "@" and "."            |
| `future_dob`               | `date_of_birth` should not be in the future |
| `underage_users`           | Users must be 18+                           |
| `future_registration_date` | Should not be in the future                 |
| `reg_before_dob`           | Registration date can't be before DOB       |
| `invalid_kyc_status`       | Must be one of: Pending, Verified, Rejected |
| `invalid_account_status`   | Must be one of: Active, Suspended, Closed   |



In [98]:
# Define valid categorical values
valid_kyc_status = ["pending", "verified", "rejected"]
valid_account_status = ["active", "suspended", "closed"]

# Apply DQ rules
dq_flags = df_cleaned_users.select(
    when(col("user_id").isNull() | (length(trim(col("user_id"))) == 0), 1).otherwise(0).alias("null_user_id"),
    when(col("username").isNull(), 1).otherwise(0).alias("null_username"),
    when(~(col("email").contains("@") & col("email").contains(".")), 1).otherwise(0).alias("invalid_email"),
    when(col("date_of_birth") > current_date(), 1).otherwise(0).alias("future_dob"),
    when(datediff(current_date(), col("date_of_birth")) < (18 * 365), 1).otherwise(0).alias("underage_users"),
    when(col("registration_date") > current_date(), 1).otherwise(0).alias("future_registration_date"),
    when(col("registration_date") < col("date_of_birth"), 1).otherwise(0).alias("reg_before_dob"),
    when(~lower(trim(col("kyc_status"))).isin(valid_kyc_status) & col("kyc_status").isNotNull(), 1).otherwise(0).alias("invalid_kyc_status"),
    when(col("account_status").isNull(), 1).otherwise(0).alias("invalid_account_status")
)

# Aggregate
metrics = dq_flags.groupBy().sum().collect()[0]

# Extract counts
dq_summary = {
    "null_user_id": metrics["sum(null_user_id)"],
    "null_username": metrics["sum(null_username)"],
    "invalid_email": metrics["sum(invalid_email)"],
    "future_dob": metrics["sum(future_dob)"],
    "underage_users": metrics["sum(underage_users)"],
    "future_registration_date": metrics["sum(future_registration_date)"],
    "reg_before_dob": metrics["sum(reg_before_dob)"],
    "invalid_kyc_status": metrics["sum(invalid_kyc_status)"],
    "invalid_account_status": metrics["sum(invalid_account_status)"]
}

# Thresholds (set according to tolerance policy)
thresholds = {
    "null_user_id": 0,
    "null_username": 0,
    "invalid_email": 10,
    "future_dob": 0,
    "underage_users": 0,
    "future_registration_date": 5,
    "reg_before_dob": 5,
    "invalid_kyc_status": 0,
    "invalid_account_status": 0
}

# Print DQ Report
print("\📋 Data Quality Report:\n--------------------------")
dq_pass = True
for check, count in dq_summary.items():
    status = "OK" if count <= thresholds[check] else "❌ Failed"
    if count > thresholds[check]:
        dq_pass = False
    print(f"{check.ljust(30)} = {count:<5} → {status}")

# Final status
if dq_pass:
    print("\nAll critical DQ checks passed. Safe to proceed to Silver layer.")
else:
    print("\n⚠️ Warning: One or more critical DQ checks failed. Manual review recommended.")


StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 100, Finished, Available, Finished)

\📋 Data Quality Report:
--------------------------
null_user_id                   = 0     → OK
null_username                  = 0     → OK
invalid_email                  = 0     → OK
future_dob                     = 0     → OK
underage_users                 = 0     → OK
future_registration_date       = 0     → OK
reg_before_dob                 = 0     → OK
invalid_kyc_status             = 0     → OK
invalid_account_status         = 0     → OK

All critical DQ checks passed. Safe to proceed to Silver layer.


#### **Step 10: Data Load to Silver Lakehouse Layer**
This step writes the cleaned users data to the Silver layer of the Lakehouse using Delta format. It uses a MERGE strategy to enable incremental upserts—updating existing user preferences and inserting new ones—while avoiding duplicates.

Steps Performed:
Define Destination Path:
- Sets the location in the Lakehouse where the cleaned data will be saved:
- Files/cdp_silver/oracle/cleaned/cleaned_users.

Prepare Columns for Consistency:
- ingestion_date: Adds the current date for use as a partition column to optimize query performance and simplify incremental loads.
- load_timestamp: Adds a consistent timestamp string (formatted as YYYYMMDDHHMMSS) that represents when the data was loaded.

Check if Delta Table Exists:
- If the table already exists at the path:
- Uses Delta Lake MERGE to compare existing and new data using the unique key user_id.
- Inserts only new (non-matching) records.

If the table does not exist:
- Creates the table with partitioning by ingestion_date for efficient querying and file organization.


In [99]:
# Define the Silver layer path
silver_path = "Files/cdp_silver/oracle/cleaned/cleaned_users"

# Add ingestion_date and load_timestamp columns
df_cleaned_users = df_cleaned_users.withColumn("ingestion_date", current_date())

load_ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
df_cleaned_users = df_cleaned_users.withColumn("load_timestamp", lit(load_ts))

# Ensure date_of_birth and registration_date are strings in "yyyy-MM-dd" format
df_cleaned_users = df_cleaned_users.withColumn(
    "date_of_birth", date_format(col("date_of_birth"), "yyyy-MM-dd").cast(StringType())
).withColumn(
    "registration_date", date_format(col("registration_date"), "yyyy-MM-dd").cast(StringType())
)

# Optional: Print schema to verify before write
df_cleaned_users.printSchema()

# Load or create the Delta table
if DeltaTable.isDeltaTable(spark, silver_path):
    delta_table = DeltaTable.forPath(spark, silver_path)

    (
        delta_table.alias("target")
        .merge(
            df_cleaned_users.alias("source"),
            "target.user_id = source.user_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        df_cleaned_users
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true") 
        .partitionBy("ingestion_date")
        .save(silver_path)
    )

print("Cleaned users loaded to Silver layer.")

StatementMeta(, bf338ae1-ebec-40e0-8d7b-7979ae64bd79, 101, Finished, Available, Finished)

root
 |-- user_id: string (nullable = true)
 |-- username: string (nullable = true)
 |-- email: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- date_of_birth: string (nullable = true)
 |-- registration_date: string (nullable = true)
 |-- country_code: string (nullable = true)
 |-- kyc_status: string (nullable = true)
 |-- account_status: string (nullable = true)
 |-- load_timestamp: string (nullable = false)
 |-- ingestion_date: date (nullable = false)

Cleaned users loaded to Silver layer.
